In [5]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage

# from evaluation.evaluation import classify_situation_node
# from response.response import (
#     clarification_response_node,
#     conflict_response_node,
#     evaluate_conditions_and_answers_node,
#     violence_response_node,
# )
from utils.llm.llm_state import ChatbotState
from utils.classes import Condition, Situation
from utils.llm.llm_prompts import situation_evaluation_prompt, condition_evaluation_prompt
from utils.llm.llm_config import llm
from data.data_loader import load_violence_criteria, violence_titles, load_conditions_criteria, load_answers, condition_titles
from utils.graph import graph

### Initialization

In [7]:
initial_message = "Je me demande si j ai subi des années de maltraitance psychologique. Mon mari avait des accès de colère durant lesquelles je il me rabaissait en me disant que j étais nulle que je savais rien faire, pas même m occuper des enfants. Il cassait des objets, jetait des objets, refusait de me parler jusqu'à 1 semaine, partait quelques heures sans que je sache où il était et coupait son téléphone. Aujourd hui je l ai quitté mais me sens mal, il me ment, me dit que je fais des histoires pour rien, que c est de nouveau de sa faute. Du coup, je me demande si le problème c est moi ou s'il me manipule et si ce que j ai vécu c est de la maltraitance."

initial_message = "Bonjour actuellement en recherche d'un emploi avec le chômage qui se termine a la fin du mois j'ai 2 enfant 6 et 3ans .avec mon conjoint le papa ce n'est plus vivable je suis dénigrer me parle mal me dit régulièrement que je vais tout perdre ainsi que les enfants que façon je je fais jamais d'effort il parle sur un ton agressif en permanence et je recent que c'est moi le problème pour tout .. que faire sans moyen .."


state: ChatbotState = {
    "messages": [HumanMessage(content=(initial_message))],
    "language": "français",  # graph nodes expect the full language name
}

config = {
    "configurable": {
        "thread_id": "debug-session-1",
    }
}

for current_state in graph.stream(
    state,
    config=config,
    stream_mode="values",
):
    print("Messages:", current_state.get("messages"))
    print("Situation:", current_state.get("situation"))
    print("Conditions:", current_state.get("conditions"))
    print("Response:", current_state.get("response"))
    print("-" * 80)

Deserializing unregistered type utils.classes.Situation from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('utils.classes', 'Situation')]
Deserializing unregistered type utils.classes.Condition from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('utils.classes', 'Condition')]


Messages: [HumanMessage(content="Je me demande si j ai subi des années de maltraitance psychologique. Mon mari avait des accès de colère durant lesquelles je il me rabaissait en me disant que j étais nulle que je savais rien faire, pas même m occuper des enfants. Il cassait des objets, jetait des objets, refusait de me parler jusqu'à 1 semaine, partait quelques heures sans que je sache où il était et coupait son téléphone. Aujourd hui je l ai quitté mais me sens mal, il me ment, me dit que je fais des histoires pour rien, que c est de nouveau de sa faute. Du coup, je me demande si le problème c est moi ou s'il me manipule et si ce que j ai vécu c est de la maltraitance.", additional_kwargs={}, response_metadata={}, id='afba5c82-c0df-42cd-ae5b-1fd364c26599'), AIMessage(content="Ce que vous avez vécu correspond à de la maltraitance psychologique, et ce n'est pas de votre faute. Votre sentiment de malaise et de doute après avoir quitté la situation est une réaction courante face à une man

### LLM Helpers

In [3]:
def violence_criteria_concatenation() -> str:
    """
    Concatenates the violence criteria from the CSV file.
    
    Returns:
        str: The concatenated violence criteria.
    """
    violence_criteria = load_violence_criteria()
    violence_text = ""
    for title, criterion in zip(violence_titles, violence_criteria["criteria"]):
        violence_text += f"## {title}:\n{criterion}\n\n"
    return violence_text.rstrip()

### Evaluation

In [ ]:
def classify_situation_node(state: ChatbotState) -> dict:
    """
    Classifies the complete conversation, including answers to follow-up questions.
    """
    prompt = ChatPromptTemplate.from_template(situation_evaluation_prompt)
    evaluator_llm = llm.with_structured_output(Situation)
    situation = (prompt | evaluator_llm).invoke(
        {
            "messages": state["messages"],
            "previous_evaluation": state.get("situation"),
            "pending_questions": state.get("pending_questions", []),
            "violence_criteria": violence_criteria_concatenation(),
        }
    )
    print(f"Situation evaluation: {situation}\n Clarification needed: {situation.needs_clarification}\n Clarification reason: {situation.clarification_reason}")

    return {
        "situation": situation,
        "needs_clarification": situation.needs_clarification,
        "clarification_reason": situation.clarification_reason,
    }

### Response Generation